In [2]:
from market import market
m1 = market(5,100,2)
m1.price([2,17])

5

In [ ]:
import numpy as np

def action(tech,capital,info,qtable):
    ep = np.random.uniform(0,1)
    if ep<0.7:
        return np.argmax(qtable)
    else:
        pass
    return 1.1 
    
def demand_function():
    pass





In [ ]:
def dice(p) -> int:
    if p < 0 or p > 1:
        raise ValueError("Not prob")
    m = np.random.uniform(0,1)
    while p < m:
        return 1
    return 0

In [4]:
while 1:
    print("true bro")
    break

true bro


In [ ]:
import random
import numpy as np
from itertools import product

# NK landscape
def build_nk(N, K, seed=None):
    rng = random.Random(seed)
    # interaction indices for each i
    neighbors = [sorted([(i + j + 1) % N for j in range(K)]) for i in range(N)]
    # random payoff of 0..1 for each local configuration
    table = [ {tuple(conf): rng.random() for conf in product([0,1], repeat=K+1)} for _ in range(N)]
    return neighbors, table

def fitness(x, neighbors, table):
    total = 0.0
    N = len(x)
    for i in range(N):
        conf = tuple([x[i]] + [x[j] for j in neighbors[i]])
        total += table[i][conf]
    return total / N

def mutate(x, action):
    y = x.copy()
    y[action] = 1 - y[action]
    return y

# Q-learning
def train_q_learning(N=10, K=2, episodes=5000, alpha=0.1, gamma=0.9, eps=0.1, seed=0):
    neighbors, table = build_nk(N, K, seed)
    # Q[state_str][action]
    Q = {}
    def state_key(x): return ''.join(str(bit) for bit in x)

    # init random state
    x = [random.randint(0,1) for _ in range(N)]
    best = fitness(x, neighbors, table)

    for ep in range(episodes):
        if ep % 500 == 0:  # for debug
            print(f"episode {ep}, best fitness {best:.4f}")
        x = [random.randint(0,1) for _ in range(N)]
        s = state_key(x)
        for step in range(2*N):
            if s not in Q: Q[s] = np.zeros(N)
            if random.random() < eps:
                a = random.randrange(N)
            else:
                a = int(np.argmax(Q[s]))
            x2 = mutate(x, a)
            s2 = state_key(x2)
            f1 = fitness(x, neighbors, table)
            f2 = fitness(x2, neighbors, table)
            r = f2 - f1
            if s2 not in Q: Q[s2] = np.zeros(N)
            Q[s][a] += alpha * (r + gamma * np.max(Q[s2]) - Q[s][a])
            x, s = x2, s2
            if f2 > best: best = f2
    return best, neighbors, table, Q

best, _, _, _ = train_q_learning()
print("best found fitness:", best)

In [4]:
import gymnasium as gym
from gymnasium import spaces

class LineWorldEnv(gym.Env):
    """A tiny custom Gym environment."""
    metadata = {"render_modes": ["human"]}

    def __init__(self, length=7):
        super().__init__()
        self.length = length
        self.action_space = spaces.Discrete(2)          # 0=left, 1=right
        self.observation_space = spaces.Discrete(length) # position 0..length-1
        self.start_pos = length // 2
        self.goal_pos = length - 1
        self.fail_pos = 0
        self.state = self.start_pos

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.state = self.start_pos
        return self.state, {}

    def step(self, action):
        if action == 0:
            self.state = max(0, self.state - 1)
        else:
            self.state = min(self.length - 1, self.state + 1)

        terminated = self.state in (self.goal_pos, self.fail_pos)
        reward = 1.0 if self.state == self.goal_pos else -1.0 if self.state == self.fail_pos else -0.01
        truncated = False
        return self.state, reward, terminated, truncated, {}

    def render(self):
        cells = ["-"] * self.length
        cells[self.goal_pos] = "G"
        cells[self.fail_pos] = "F"
        cells[self.state] = "A"
        print("".join(cells))


# Quick Q-learning example
env = LineWorldEnv(length=9)
Q = np.zeros((env.observation_space.n, env.action_space.n))

alpha, gamma = 0.1, 0.95
eps = 0.2
episodes = 500

for _ in range(episodes):
    s, _ = env.reset()
    done = False
    while not done:
        if np.random.rand() < eps:
            a = env.action_space.sample()
        else:
            a = int(np.argmax(Q[s]))

        s2, r, terminated, truncated, _ = env.step(a)
        done = terminated or truncated

        Q[s, a] += alpha * (r + gamma * np.max(Q[s2]) - Q[s, a])
        s = s2

# Test learned policy
s, _ = env.reset()
done = False
total_reward = 0
while not done:
    a = int(np.argmax(Q[s]))
    s, r, terminated, truncated, _ = env.step(a)
    env.render()
    total_reward += r
    done = terminated or truncated

print("Total reward:", total_reward)

F----A--G
F-----A-G
F------AG
F-------A
Total reward: 0.97


In [ ]:
class simulate:
    def __init__(self, alpha, gamma, eps, epi, max_steps, env) -> None:
        '''
        alpha: stepwise\\
        gamma:discount rate\\
        eps: greedy para\\
        epi: mont carlo para\\
        max_steps: in an epsisode\\
        env: environment object should have all info we need\\
        '''
        self.alpha = alpha
        self.gamma = gamma
        self.eps = eps
        self.epi = epi
        self.max_steps = max_steps
        self.env = env

    def begin(self):
        Q = np.zeros((self.env.n, self.env.num_states, self.env.num_actions))
        actions = np.zeros((self.max_steps, self.env.n))
        memo = np.zeros((self.max_steps, self.env.n))
        for epis in range(self.epi):
            ienv = self.env
            done = False
            step = 0
            
            incre_memo = np.array([])
            incre_Q = np.zeros(Q.shape)
            incre_action = np.array([])
            
            e = np.zeros((self.max_steps,self.env.n))

            while not done and step < self.max_steps:
                max_e = np.minimum(np.asarray(ienv.k), np.asarray(ienv.input_limit))
                e_levels = np.linspace(1e-10, max_e, num=self.env.num_actions)
                state = ienv.new_tech()
                # agent decisions
                for _ in range(actions.shape[1]):
                    
                    if dice(eps):
                        # actions[step][_] = np.random.randint(self.env.num_actions)
                        action = np.random.randint(self.env.num_actions)
                    else:
                        action = np.argmax(incre_Q[_, state[_]])

                    actions[step][_] = action
                    e[step][_] = e_levels[action][_]

                # 数据收集
                reward = ienv.session(e)
                incre_memo = np.append(incre_memo,reward)
                incre_action = np.append(incre_action, actions[step])
                
                # 更新环境
                ienv.update_tech(e)
                ienv.update_q(incre_Q)
                ienv.update_agent()
                
                step += 1

            
            memo = (epis/(1+epis))*memo + incre_memo/(epis+1)
            actions = (epis/(1+epis))*actions + incre_action/(epis+1)
            Q = (epis/(1+epis))*Q + incre_Q/(epis+1)
    
        return actions, memo, Q
        






def update(capitals,actions,techs,demand_function,cost,output,invest,transform):
    price = demand_function(sum(actions))
    profit = [price*output(actions[i])-cost(techs[i],actions[i])-invest(actions[i]) for i in range(len(actions))]
    techs = transform(techs, actions,capitals)
    capitals += profit
    return capitals, techs

In [2]:
import numpy as np
np.zeros((4,3,3))

array([[[0., 0., 0.],
        [0., 0., 0.],
        [0., 0., 0.]],

       [[0., 0., 0.],
        [0., 0., 0.],
        [0., 0., 0.]],

       [[0., 0., 0.],
        [0., 0., 0.],
        [0., 0., 0.]],

       [[0., 0., 0.],
        [0., 0., 0.],
        [0., 0., 0.]]])

In [6]:
np.linspace(1e-10, (1,6,2),10)
# (1,3,2)

array([[1.00000000e-10, 1.00000000e-10, 1.00000000e-10],
       [1.11111111e-01, 6.66666667e-01, 2.22222222e-01],
       [2.22222222e-01, 1.33333333e+00, 4.44444445e-01],
       [3.33333333e-01, 2.00000000e+00, 6.66666667e-01],
       [4.44444444e-01, 2.66666667e+00, 8.88888889e-01],
       [5.55555556e-01, 3.33333333e+00, 1.11111111e+00],
       [6.66666667e-01, 4.00000000e+00, 1.33333333e+00],
       [7.77777778e-01, 4.66666667e+00, 1.55555556e+00],
       [8.88888889e-01, 5.33333333e+00, 1.77777778e+00],
       [1.00000000e+00, 6.00000000e+00, 2.00000000e+00]])